# Deploying DeepSeek-V4-Flash with vLLM: GitHub Repository Analysis

This notebook will walk you through how to run the [`deepseek-ai/DeepSeek-V4-Flash`](https://huggingface.co/deepseek-ai/DeepSeek-V4-Flash) model on **2× NVIDIA B200s** using [vLLM](https://github.com/vllm-project/vllm), and use its 1M-token context window to analyze an entire open-source GitHub repository.

[vLLM](https://github.com/vllm-project/vllm) is a fast and easy-to-use library for LLM inference and serving.

For more details on the model, [click here](https://huggingface.co/deepseek-ai/DeepSeek-V4-Flash).

Prerequisites:
- 2× NVIDIA B200 (192 GB HBM3e each, 384 GB total) — model weights are ~160 GB; two GPUs provide the necessary headroom for the KV cache and activations
- Docker (recommended) or a bare-metal environment with the full CUDA 13 toolkit
- A Hugging Face account with access to [`deepseek-ai/DeepSeek-V4-Flash`](https://huggingface.co/deepseek-ai/DeepSeek-V4-Flash)
- Git

## Overview

- **Serve** DeepSeek-V4-Flash on 2× NVIDIA B200s using vLLM
- **Load** a full open-source GitHub repository into the model's context window
- **Generate** an architecture map, onboarding guide, and code navigation answers
- **Compare** Non-Think and Think-High reasoning modes on a complex refactoring question

## Table of Contents

1. **Install dependencies** — client packages
2. **Start the vLLM server** — Docker (recommended) or bare-metal
3. **Load the GitHub repository** — shallow clone, file collection, context assembly
4. **Architecture map** — system overview, components, request lifecycle
5. **Onboarding guide** — reading order, mental model, first contributions
6. **"Where would I change X?"** — targeted code navigation
7. **Non-Think vs Think-High** — reasoning mode comparison

## Step 0 — Install Dependencies

In [ ]:
%pip install --quiet --upgrade openai gitpython requests

In [ ]:
import os

# Paste your token here if you haven't run `huggingface-cli login`
# os.environ["HF_TOKEN"] = "hf_..."

## Step 1 — Start the vLLM Server

Run the following command in a separate terminal before continuing. Leave the server running while this notebook executes.

### Option A — Docker (recommended)

The `vllm/vllm-openai` image includes the full CUDA 13 toolkit, so FP8/FP4 JIT kernels (FlashInfer, DeepGEMM) compile correctly without requiring `nvcc` on the host.

```bash
docker run --gpus '"device=0,1"' \
  --ipc=host -p 8000:8000 \
  --network host \
  -v ~/.cache/huggingface:/root/.cache/huggingface \
  vllm/vllm-openai:latest \
  deepseek-ai/DeepSeek-V4-Flash \
  --trust-remote-code \
  --kv-cache-dtype fp8 \
  --block-size 256 \
  --tensor-parallel-size 2 \
  --max-model-len 131072 \
  --gpu-memory-utilization 0.90 \
  --tokenizer-mode deepseek_v4 \
  --reasoning-parser deepseek_v4
```

> **Note:** `~/.cache/huggingface` is mounted into the container so that previously downloaded model weights are reused automatically.

### Option B — pip / uv (bare-metal)

For bare-metal installs, follow the hardware-specific recipe at [recipes.vllm.ai/deepseek-ai/DeepSeek-V4-Flash](https://recipes.vllm.ai/deepseek-ai/DeepSeek-V4-Flash?hardware=b200). This requires a full CUDA toolkit (`nvcc` and dev headers) on the host. The same flags apply.

### Configuration reference

| Flag | Purpose |
|---|---|
| `--kv-cache-dtype fp8` | Required for DeepSeek-V4's MLA attention; `auto` will error |
| `--tensor-parallel-size 2` | Splits the ~160 GB model across 2× B200 GPUs |
| `--max-model-len 131072` | 128K context — increase up to 1M if needed |
| `--gpu-memory-utilization 0.90` | Reserves 10% headroom for CUDA operations |
| `--block-size 256` | Recommended block size for DeepSeek-V4's MLA KV cache |
| `--tokenizer-mode deepseek_v4` | Required for DeepSeek-V4's custom tokenizer |
| `--reasoning-parser deepseek_v4` | Parses `<think>...</think>` blocks and exposes `reasoning_content` in API responses |

> **Expected output:** Wait for `INFO: Application startup complete` before continuing. On first run, model weights will be downloaded from Hugging Face — this may take several minutes.

In [1]:
import requests, time

VLLM_BASE_URL = "http://localhost:8000"

for attempt in range(30):
    try:
        if requests.get(f"{VLLM_BASE_URL}/health", timeout=5).status_code == 200:
            print("Server ready.")
            break
    except requests.exceptions.ConnectionError:
        pass
    print(f"Waiting... ({attempt + 1}/30)")
    time.sleep(10)

Server ready.


In [2]:
from openai import OpenAI

client = OpenAI(base_url=f"{VLLM_BASE_URL}/v1", api_key="not-needed")
MODEL  = "deepseek-ai/DeepSeek-V4-Flash"

print([m.id for m in client.models.list().data])

['deepseek-ai/DeepSeek-V4-Flash']


## Step 2 — Load the GitHub Repository

This notebook uses [`vllm-project/vllm`](https://github.com/vllm-project/vllm) as the target repository. A shallow clone (`--depth 1`) retrieves the latest snapshot without pulling the full commit history.

- Individual files are capped at 15 KB to avoid loading single large files
- Total context is capped at ~80K tokens — increase `MAX_CONTEXT_CHARS` up to ~4M characters to use the full 1M-token window
- Test directories and build artifacts are skipped

In [3]:
import subprocess, pathlib

REPO_URL  = "https://github.com/vllm-project/vllm"
CLONE_DIR = pathlib.Path("./vllm_repo")

if not CLONE_DIR.exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(CLONE_DIR)], check=True)

Cloning into 'vllm_repo'...
Updating files: 100% (5351/5351), done.


In [4]:
MAX_FILE_CHARS    = 15_000
MAX_CONTEXT_CHARS = 320_000  # ~80K tokens

INCLUDE_EXTS = {".py", ".md", ".rst", ".toml", ".cfg", ".yaml", ".yml", ".txt"}
SKIP_DIRS    = {"__pycache__", ".git", ".github", "node_modules", ".eggs",
                "dist", "build", ".mypy_cache", ".pytest_cache", "benchmarks"}

PRIORITY_GLOBS = [
    "README.md", "CONTRIBUTING.md", "docs/**/*.md",
    "vllm/__init__.py", "vllm/engine/*.py", "vllm/entrypoints/*.py",
    "vllm/model_executor/models/*.py", "vllm/core/*.py", "vllm/worker/*.py",
]


def iter_priority_files(root):
    seen = set()
    for pattern in PRIORITY_GLOBS:
        for p in sorted(root.glob(pattern)):
            if p.is_file() and p not in seen:
                seen.add(p)
                yield p
    for p in sorted(root.rglob("*")):
        if (
            p.is_file() and p not in seen
            and p.suffix in INCLUDE_EXTS
            and not any(d in p.parts for d in SKIP_DIRS)
            and "test" not in p.parts
        ):
            seen.add(p)
            yield p


def build_tree(root, max_depth=3):
    lines = []
    def _walk(path, depth):
        if depth > max_depth:
            return
        for child in sorted(path.iterdir()):
            if child.name.startswith(".") or child.name in SKIP_DIRS:
                continue
            lines.append("  " * depth + child.name + ("/" if child.is_dir() else ""))
            if child.is_dir():
                _walk(child, depth + 1)
    _walk(root, 0)
    return "\n".join(lines)


context_parts  = [f"# Repository: vllm-project/vllm\n\n## Directory Tree\n\n```\n{build_tree(CLONE_DIR)}\n```"]
total_chars    = len(context_parts[0])
files_included = []

for fpath in iter_priority_files(CLONE_DIR):
    if total_chars >= MAX_CONTEXT_CHARS:
        break
    try:
        text = fpath.read_text(encoding="utf-8", errors="ignore")
    except OSError:
        continue
    if not text.strip() or len(text) > MAX_FILE_CHARS:
        continue
    rel = fpath.relative_to(CLONE_DIR)
    section = f"\n\n## File: {rel}\n\n```\n{text.rstrip()}\n```"
    context_parts.append(section)
    total_chars += len(section)
    files_included.append(str(rel))

repo_context = "\n".join(context_parts)
print(f"{len(files_included)} files  |  {total_chars:,} chars  |  ~{total_chars // 4:,} tokens")

80 files  |  321,811 chars  |  ~80,452 tokens


---
## Step 3 — Architecture Map

In [5]:
ARCH_PROMPT = """\
Here is the vllm-project/vllm repository:

{repo_context}

---

Produce a concise Architecture Map with these sections:

1. **System Overview** (3–5 sentences): What vLLM does and its design goals.
2. **Component Map**: Major modules/packages with one-line descriptions.
3. **Request Lifecycle**: How an API request flows from HTTP entrypoint to GPU kernel and back.
4. **Key Abstractions**: The most important classes a new contributor must understand (with file paths).
5. **Extension Points**: Where the codebase is designed for extensibility.
"""

response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": ARCH_PROMPT.format(repo_context=repo_context)}],
    max_tokens=2500,
    temperature=0.2,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)
print(response.choices[0].message.content)

# vLLM Architecture Map

## 1. System Overview

vLLM is a high-throughput, memory-efficient LLM inference and serving library. Its design centers on **PagedAttention**, which manages KV cache in non-contiguous blocks to eliminate fragmentation and enable near-100% memory utilization. The system achieves state-of-the-art serving throughput through continuous batching, chunked prefill, prefix caching, and optimized CUDA/HIP kernels. vLLM supports distributed inference across tensor, pipeline, data, expert, and context parallelism, and provides OpenAI-compatible API servers, gRPC endpoints, and an offline inference API.

## 2. Component Map

| Package | Description |
|---------|-------------|
| `vllm/entrypoints/` | HTTP/gRPC API servers (OpenAI-compatible, Anthropic, gRPC) |
| `vllm/engine/` | Core engine: `LLMEngine` (offline), `AsyncLLMEngine` (online) |
| `vllm/v1/engine/` | Next-gen engine with disaggregated scheduling and core-client architecture |
| `vllm/config/` | All configurati

---
## Step 4 — Onboarding Guide for New Contributors

In [6]:
ONBOARDING_PROMPT = """\
Here is the vllm-project/vllm repository:

{repo_context}

---

Write a New Contributor Onboarding Guide covering:

1. **Mental model**: The 5 most important concepts to understand first.
2. **Reading order**: Which files to read, in what order, and why.
3. **Local dev setup**: Running the test suite and a minimal local server.
4. **First contributions**: 3 specific, self-contained areas to make a real improvement.
5. **Common pitfalls**: Mistakes that trip up new contributors most often.
"""

response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": ONBOARDING_PROMPT.format(repo_context=repo_context)}],
    max_tokens=2500,
    temperature=0.2,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)
print(response.choices[0].message.content)

# New Contributor Onboarding Guide

Welcome to vLLM! This guide will help you get productive quickly by focusing on what matters most for new contributors.

---

## 1. Mental Model: The 5 Most Important Concepts

### 1.1 The Engine-Scheduler-Model Pipeline

vLLM's core loop is simple in concept:

```
Request → Scheduler → Model Runner → Output
```

- **Scheduler**: Decides which requests to batch together, manages KV cache allocation
- **Model Runner**: Executes the model forward pass with the scheduled batch
- **Engine**: Orchestrates the loop, handles async I/O

The key insight: vLLM is **not** a simple `model.forward()` wrapper. It's a stateful serving system that manages memory, batching, and parallelism.

### 1.2 PagedAttention

The foundational innovation. Instead of allocating contiguous KV cache blocks per request, vLLM uses **paged** blocks (like virtual memory). This eliminates fragmentation and enables:

- **Prefix caching**: Reuse KV cache across requests with shared prefix

## Step 5 — "Where Would I Change X?"

Edit `QUESTIONS` to ask about changes relevant to your own project.

In [7]:
QUESTIONS = [
    "Where would I add support for a new quantization format (e.g., INT4 GPTQ)?",
    "How would I add a new sampling strategy beyond top-p and top-k?",
    "Where is the batching logic and how would I change the batching policy?",
]

for i, question in enumerate(QUESTIONS, 1):
    print(f"\nQ{i}: {question}")
    print("─" * 60)
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "Answer with exact file paths, class names, and function names. Be concise."},
            {"role": "user",   "content": f"Repository:\n\n{repo_context}\n\nQuestion: {question}"},
        ],
        max_tokens=1000,
        temperature=0.2,
        extra_body={"chat_template_kwargs": {"enable_thinking": False}},
    )
    print(response.choices[0].message.content)


Q1: Where would I add support for a new quantization format (e.g., INT4 GPTQ)?
────────────────────────────────────────────────────────────
To add support for a new quantization format (e.g., INT4 GPTQ), you would need to modify files in these key areas:

1. **Quantization configuration**: `vllm/config/quantization.py` - Add a new config class for your format
2. **Quantization kernel implementation**: `csrc/quantization/` - Add CUDA kernels (e.g., `csrc/quantization/gptq/` for GPTQ-specific kernels)
3. **Quantization layer**: `vllm/model_executor/layers/quantization/` - Add a new directory and implementation (e.g., `vllm/model_executor/layers/quantization/gptq/`)
4. **Model layer integration**: Modify `vllm/model_executor/layers/linear.py` to support the new quantization method in linear layers
5. **Weight loading**: Update `vllm/model_executor/model_loader/weight_utils.py` to handle the new format's weight format
6. **Tests**: Add tests in `tests/quantization/` (e.g., `tests/quantiza

## Step 6 — Non-Think vs Think-High

DeepSeek-V4-Flash supports three reasoning modes. The Think-High response includes a `<think>...</think>` block containing the model's internal reasoning, which is useful for auditing complex decisions.

| Mode | Triggered by | When to use |
|---|---|---|
| Non-Think | `enable_thinking=False` in `extra_body` | Quick lookups, summaries |
| Think-High | `enable_thinking=True` in `extra_body` | Complex refactors, architecture decisions |
| Think-Max | `"reasoning_effort": "max"` in `chat_template_kwargs` ([see model card](https://huggingface.co/deepseek-ai/DeepSeek-V4-Flash)) | Most demanding reasoning tasks |

> **Note:** `--reasoning-parser deepseek_v4` must be set on the server. Reasoning content is returned in `choices[0].message.reasoning_content`; if that field is absent, check `choices[0].message.model_extra` (field name varies by vLLM version).

In [8]:
HARD_QUESTION = (
    "I want to replace vLLM's current FCFS scheduler with a deadline-aware scheduler "
    "that prioritizes requests with an SLA deadline field. "
    "Which classes and files would I need to modify or create, in what order, "
    "and what are the key invariants I must preserve to avoid breaking the paged attention memory manager?"
)

base_message = [{"role": "user", "content": f"Repository:\n\n{repo_context}\n\nQuestion: {HARD_QUESTION}"}]

resp_nothink = client.chat.completions.create(
    model=MODEL,
    messages=base_message,
    max_tokens=3000,
    temperature=0.2,
    extra_body={"chat_template_kwargs": {"enable_thinking": False}},
)

resp_think = client.chat.completions.create(
    model=MODEL,
    messages=base_message,
    max_tokens=4000,
    temperature=0.6,
    extra_body={"chat_template_kwargs": {"enable_thinking": True}},
)

In [9]:
print("NON-THINK")
print("─" * 60)
print(resp_nothink.choices[0].message.content)

think_msg = resp_think.choices[0].message
reasoning = (
    getattr(think_msg, "reasoning_content", None)
    or (think_msg.model_extra or {}).get("reasoning_content")
    or (think_msg.model_extra or {}).get("reasoning")
)

print("\n\nTHINK-HIGH")
print("─" * 60)
if reasoning:
    print(f"[reasoning — {len(reasoning.split()):,} words]\n")
    print(reasoning)
    print("\n[answer]\n")
print(think_msg.content)

reasoning_words = len(reasoning.split()) if reasoning else 0
answer_words    = len((think_msg.content or "").split())

print(f"\nTOKEN SUMMARY")
print(f"  Non-think  completion tokens : {resp_nothink.usage.completion_tokens:>6,}")
print(f"  Think-High completion tokens : {resp_think.usage.completion_tokens:>6,}")
print(f"    ├─ reasoning block (~words): {reasoning_words:>6,}")
print(f"    └─ final answer  (~words)  : {answer_words:>6,}")
print(f"  Extra tokens spent thinking  : {resp_think.usage.completion_tokens - resp_nothink.usage.completion_tokens:>+6,}")

NON-THINK
────────────────────────────────────────────────────────────
# Deadline-Aware Scheduler for vLLM

Based on the repository structure, here's a comprehensive plan to implement a deadline-aware scheduler while preserving the PagedAttention memory manager invariants.

## Files to Modify (in order)

### 1. **Core Scheduler Interface** - `vllm/v1/core/sched/`
First, understand the existing scheduler structure:

- `vllm/v1/core/sched/` - Contains scheduler implementations
- `vllm/v1/core/sched/__init__.py` - Scheduler exports

### 2. **Scheduler Base Class** - `vllm/v1/core/sched/base.py` (likely exists or needs creation)

### 3. **Existing Scheduler** - `vllm/v1/core/sched/fcfs.py` (or similar name)
This is the current FCFS implementation you'll use as a reference.

### 4. **Request Model** - `vllm/v1/request.py`
Add the `deadline` field to the request class.

### 5. **Scheduler Config** - `vllm/config/scheduler.py`
Add deadline-related configuration parameters.

### 6. **Engine Co